# WiNDC ↔ OECD ICIO correspondence table

This notebook builds the concordance table that links the **71 WiNDC sectors** to the **50 OECD ICIO sectors** (based on ISIC Rev. 4).

**Correspondence chain:**
```
WiNDC  →  BEA Summary  →  NAICS US 2017  →  NAICS US 2012  →  ISIC Rev. 4  →  OECD ICIO
```

**Sources:**
| Table | File |
|---|---|
| OECD ICIO ↔ ISIC Rev. 4 | `corr_OECDICIO_ISICRev4.csv` |
| US NAICS 2012 ↔ ISIC Rev. 4 | `ISIC4-NAICS2012US.txt` (UN Statistics Division) |
| BEA Summary ↔ NAICS US 2017 | `BEA-Industry-and-Commodity-Codes-and-NAICS-Concordance.xlsx` |
| WiNDC ↔ BEA Summary | `windc_base.gdx` (set `i`, attribute `assoc_texts`) |

In [ ]:
from paths import ROOT
import pandas as pd
import re
from gdx2py import GdxFile
import gamspy_base

Final structure (24 cells):

Section	Cells
Title + introduction	0
Imports	1
§1 — Sources: OECD/ISIC, US NAICS 2012/ISIC, BEA/NAICS, WiNDC/BEA	2–10
§2 — Construction: WiNDC→NAICS join, NAICS→ISIC→OECD mapping, final matching	11–17
§3 — Results: coverage stats, top sectors, unmatched	18–21
Final table + CSV export	22–23
Final correspondence table (data/interim/windc_oecd_concordance.csv):

207 rows — 71 WiNDC sectors × their OECD codes (some have several codes reflecting a real ambiguity in the NAICS/ISIC concordance)
71/73 WiNDC sectors matched (oth and use have no standard OECD ICIO equivalent)
50/50 OECD codes covered
Columns: windc_sector, windc_desc, bea_summary, bea_summary_desc, oecd_code, OECD ICIO Industry

# 1. Source correspondence tables

## 1.1 OECD ICIO ↔ ISIC Rev. 4

The OECD ICIO classification aggregates ISIC Rev. 4 into **50 sectors**. Some OECD codes cover several ISIC divisions (e.g. `C10T12` = divisions 10, 11, 12).

In [ ]:
corr_OECDICIO_ISICRev4 = pd.read_csv(
    str(ROOT / 'data/raw/correspondence/corr_OECDICIO_ISICRev4.csv')
)   
corr_OECDICIO_ISICRev4[['OECD ICIO Code (count = 50)', 'OECD ICIO Industry', 'ISIC Rev4 code']].drop_duplicates().head(10)

## 1.2 US NAICS 2012 ↔ ISIC Rev. 4

Concordance table published by the United Nations Statistics Division. Each 6-digit US NAICS 2012 code is associated with a 4-digit ISIC Rev. 4 code. 1,065 NAICS codes covered, of which only 1 is unmapped (`999001` = extraterritorial activities).

In [ ]:
naics_ISIC4 = pd.read_csv(
    str(ROOT / 'data/raw/correspondence/ISIC4-NAICS2012US.txt'),
    quotechar='"', dtype=str
)
naics_ISIC4.columns = [c.strip() for c in naics_ISIC4.columns]
naics_ISIC4 = naics_ISIC4.assign(
    naics_us6=naics_ISIC4['NAICS2012Code'].astype(str).str.strip(),
    ISIC4Code=naics_ISIC4['ISIC4Code'].astype(str).str.strip(),
)
naics_ISIC4[['naics_us6', 'ISIC4Code']].head(5)

## 1.3 BEA Summary ↔ NAICS US 2017

BEA concordance linking the BEA summary codes to the NAICS US 2017 codes. The `Summary` column contains mixed types (integers for purely numeric codes like `211`, `42`; strings for alphanumeric codes like `111CA`, `113FF`) — it is normalized to string on read.

In [ ]:
BEAsummary_NAICS = pd.read_excel(
    str(ROOT / 'data/raw/correspondence/BEA-Industry-and-Commodity-Codes-and-NAICS-Concordance.xlsx'),
    sheet_name='NAICS Codes',
    header=4
)
BEAsummary_NAICS.head()

BEAsummary_NAICS['naics_bea'] = (
    BEAsummary_NAICS['Related 2017 NAICS Codes']
    .astype(str).str.strip().str.replace(r'\*', '', regex=True)
)
BEAsummary_NAICS = (BEAsummary_NAICS
    .dropna(subset=['Related 2017 NAICS Codes'])
    .rename(columns={
        'Summary':       'bea_summary',
        'Description.1': 'bea_summary_desc',
        'Description.5': 'naics_desc',
    })
    .assign(bea_summary=lambda df: df['bea_summary'].astype(str).str.strip())
    [['bea_summary', 'bea_summary_desc', 'naics_bea', 'naics_desc']]
    .drop_duplicates()
    .reset_index(drop=True)
)
BEAsummary_NAICS.head(5)

## 1.4 WiNDC ↔ BEA Summary

The BEA Summary code of each WiNDC sector is encoded in the sector's long name (attribute `assoc_texts` of the set `i` of the file `windc_base.gdx`), in parentheses at the end of the string, e.g. `"Farms (111CA)"`.

In [ ]:
path_windc_gdx = str(ROOT / "data/raw/GTAPWiNDC/data/core/WiNDCdatabase.gdx")

_gdx_base = GdxFile(
    path_windc_gdx.replace('WiNDCdatabase.gdx', 'windc_base.gdx'),
    gams_dir=gamspy_base.directory,
)
_s = dict(_gdx_base)['i']

WINDC_TO_SUMMARY = {
    code: re.search(r'\(([^)]+)\)$', label).group(1)
    for code, label in zip(list(_s), _s.assoc_texts)
    if re.search(r'\(([^)]+)\)$', label)
}

windc_summary_df = pd.DataFrame.from_dict(
    WINDC_TO_SUMMARY, orient='index', columns=['bea_summary']
).rename_axis('windc_sector').reset_index()

print(f"{len(windc_summary_df)} WiNDC sectors, {windc_summary_df['bea_summary'].nunique()} unique BEA Summary codes")
windc_summary_df.head(5)

# 2. Construction of the WiNDC ↔ OECD ICIO correspondence table

## 2.1 WiNDC → BEA Summary → NAICS US 2017

We join `windc_summary_df` and `BEAsummary_NAICS` on the BEA Summary code. Each WiNDC sector is assigned one or more NAICS US 2017 codes (2 to 5 digits depending on the BEA level of detail).

In [ ]:
WINDC_SUMMARY_NAICS = windc_summary_df.merge(
    BEAsummary_NAICS,
    on='bea_summary',
    how='left',
)

no_naics = WINDC_SUMMARY_NAICS[WINDC_SUMMARY_NAICS['naics_bea'].isna()]['windc_sector'].tolist()
print(f"Sectors without NAICS : {no_naics}")
print(f"Total rows      : {len(WINDC_SUMMARY_NAICS)}")
WINDC_SUMMARY_NAICS.head(5)

## 2.2 NAICS US 2012 → ISIC Rev. 4 → OECD ICIO

The UN Stats table uses **pure numeric** ISIC Rev. 4 codes (e.g. `3510`), whereas the OECD prefixes use the **letter + digits** format (e.g. `D`, `C10T12`). The conversion goes through the ISIC division → ISIC section mapping (e.g. division `35` → section `D`). The composite OECD codes (`C10T12`, `C17_18`, `C24A`…) are expanded into their constituent divisions.

In [ ]:
# ISIC4 division→section letter mapping (converts bare numeric ISIC4 codes
# from the US NAICS table, e.g. '3510', to letter-prefixed OECD format, e.g. 'D35').
_div_to_section = {}
for _d in range(1, 4):   _div_to_section[_d] = 'A'
for _d in range(5, 10):  _div_to_section[_d] = 'B'
for _d in range(10, 34): _div_to_section[_d] = 'C'
_div_to_section[35] = 'D'
for _d in range(36, 40): _div_to_section[_d] = 'E'
for _d in range(41, 44): _div_to_section[_d] = 'F'
for _d in range(45, 48): _div_to_section[_d] = 'G'
for _d in range(49, 54): _div_to_section[_d] = 'H'
for _d in range(55, 57): _div_to_section[_d] = 'I'
for _d in range(58, 64): _div_to_section[_d] = 'J'
for _d in range(64, 67): _div_to_section[_d] = 'K'
_div_to_section[68] = 'L'
for _d in range(69, 76): _div_to_section[_d] = 'M'
for _d in range(77, 83): _div_to_section[_d] = 'N'
_div_to_section[84] = 'O'
_div_to_section[85] = 'P'
for _d in range(86, 89): _div_to_section[_d] = 'Q'
for _d in range(90, 94): _div_to_section[_d] = 'R'
for _d in range(94, 97): _div_to_section[_d] = 'S'
for _d in range(97, 99): _div_to_section[_d] = 'T'
_div_to_section[99] = 'U'

def numeric_isic4_to_letter(code):
    c = str(code).strip()
    try: div = int(c[:2])
    except: return None
    section = _div_to_section.get(div, '')
    return section + c if section else None

def oecd_to_isic_prefixes(oecd_code):
    special = {'C24A': ['C241'], 'C24B': ['C242', 'C243']}
    if oecd_code in special:
        return special[oecd_code]
    if re.match(r'^[A-Z]$', oecd_code):
        return [oecd_code]
    m = re.match(r'^([A-Z])(\d+)T(\d+)$', oecd_code)
    if m:
        letter, start, end, w = m.group(1), int(m.group(2)), int(m.group(3)), len(m.group(2))
        return [f'{letter}{n:0{w}d}' for n in range(start, end + 1)]
    m = re.match(r'^([A-Z])(\d+)_(\d+)$', oecd_code)
    if m:
        return [f'{m.group(1)}{m.group(2)}', f'{m.group(1)}{m.group(3)}']
    return [oecd_code]

isic_pref_to_oecd = {}
for _, row in corr_OECDICIO_ISICRev4.iterrows():
    oecd = row['OECD ICIO Code (count = 50)']
    for prefix in oecd_to_isic_prefixes(oecd):
        isic_pref_to_oecd[prefix] = oecd

def isic_letter_to_oecd(isic_letter):
    s = str(isic_letter).strip()
    for length in range(len(s), 0, -1):
        if s[:length] in isic_pref_to_oecd:
            return isic_pref_to_oecd[s[:length]]
    return None

naics_ISIC4['isic_letter'] = naics_ISIC4['ISIC4Code'].apply(numeric_isic4_to_letter)
naics_ISIC4['oecd_code']   = naics_ISIC4['isic_letter'].apply(isic_letter_to_oecd)

naics_to_oecd = (naics_ISIC4[['naics_us6', 'oecd_code']]
    .dropna(subset=['oecd_code'])
    .drop_duplicates()
    .reset_index(drop=True))

print(f"US NAICS with OECD code: {naics_to_oecd['naics_us6'].nunique()} unique codes")
print(f"Codes OECD couverts     : {naics_to_oecd['oecd_code'].nunique()} / 50")
naics_to_oecd.head()


## 2.3 Expansion and final matching

Each BEA NAICS code (variable, 2-5 digits) is used as a **prefix** to retrieve the corresponding 6-digit NAICS US 2012 codes, which are then joined to `naics_to_oecd`. A manual supplement covers the **4 government sectors** (without a NAICS code by definition) → OECD `O`.

The `oth` (non-comparable imports) and `use` (second-hand goods/waste) sectors have no standard OECD ICIO equivalent and stay unmatched.

In [ ]:
all_us6 = naics_to_oecd['naics_us6'].unique()

def expand_bea_to_us(bea_code):
    """All US 6-digit NAICS codes that start with bea_code (max 5 digits)."""
    prefix = str(bea_code).strip()[:5]
    return [c for c in all_us6 if c.startswith(prefix)]

expanded = (WINDC_SUMMARY_NAICS[['windc_sector', 'bea_summary', 'bea_summary_desc', 'naics_bea']]
            .dropna(subset=['naics_bea'])
            .copy())
expanded['naics_us6'] = expanded['naics_bea'].apply(expand_bea_to_us)
expanded = (expanded
            .explode('naics_us6')
            .dropna(subset=['naics_us6'])
            .merge(naics_to_oecd, on='naics_us6', how='left'))

# Deduplicated WiNDC ↔ OECD pairs with descriptions
pairs = (expanded[['windc_sector', 'bea_summary_desc', 'oecd_code']]
         .dropna(subset=['oecd_code'])
         .drop_duplicates()
         .merge(corr_OECDICIO_ISICRev4[['OECD ICIO Code (count = 50)', 'OECD ICIO Industry']],
                left_on='oecd_code', right_on='OECD ICIO Code (count = 50)', how='left')
         .drop(columns='OECD ICIO Code (count = 50)')
         .reset_index(drop=True))

# Supplementary mapping: government sectors have no NAICS codes by definition.
# wht/gmt/pub/fin are now covered naturally by the US NAICS 2012 table.
MANUAL_OECD = {
    'fdd': 'O',   # Federal government defense
    'fnd': 'O',   # Federal government non-defense
    'sle': 'O',   # State/local government enterprises
    'slg': 'O',   # State/local general government
    # 'oth' (noncomparable imports) and 'use' (used goods/scrap) have no OECD equivalent
}
_sector_labels = dict(zip(list(_s), _s.assoc_texts))
_oecd_desc = corr_OECDICIO_ISICRev4[['OECD ICIO Code (count = 50)', 'OECD ICIO Industry']].drop_duplicates()
supplement = (pd.DataFrame.from_dict(MANUAL_OECD, orient='index', columns=['oecd_code'])
    .rename_axis('windc_sector').reset_index()
    .assign(bea_summary_desc=lambda df: df['windc_sector'].map(_sector_labels))
    .merge(_oecd_desc, left_on='oecd_code', right_on='OECD ICIO Code (count = 50)', how='left')
    .drop(columns='OECD ICIO Code (count = 50)')
    [['windc_sector', 'bea_summary_desc', 'oecd_code', 'OECD ICIO Industry']])
pairs = pd.concat([pairs, supplement], ignore_index=True).drop_duplicates(subset=['windc_sector', 'oecd_code'])

print(f'Paires uniques WiNDC-OECD : {len(pairs)}')
pairs.head()

# 3. Results and validation

In [ ]:
oecd_col = 'oecd_code'

no_oecd  = set(windc_summary_df['windc_sector']) - set(pairs['windc_sector'])
no_windc = set(corr_OECDICIO_ISICRev4['OECD ICIO Code (count = 50)']) - set(pairs[oecd_col])

print(f"WiNDC without OECD ({len(no_oecd)}) : {sorted(no_oecd)}")
print(f"OECD without WiNDC ({len(no_windc)}) : {sorted(no_windc)}")
print()

per_windc = (pairs.groupby('windc_sector')[oecd_col]
             .apply(list)
             .rename('oecd_codes'))
per_windc = pd.DataFrame({
    'n_oecd':     per_windc.apply(len),
    'oecd_codes': per_windc,
}).sort_values('n_oecd', ascending=False)

print("=== WiNDC -> number of unique OECD codes ===")
print(per_windc['n_oecd'].value_counts().sort_index().rename_axis('n_oecd').to_frame('n_windc_sectors'))
print()
print("=== OECD -> number of unique WiNDC codes ===")
per_oecd = pairs.groupby(oecd_col)['windc_sector'].nunique().rename('n_windc')
print(per_oecd.value_counts().sort_index().rename_axis('n_windc').to_frame('n_oecd_codes'))


In [ ]:
n_top      = 10    # number of WiNDC sectors to display
show_codes = True  # True = displays the list of OECD codes

print(f"Top {n_top} WiNDC by number of OECD codes:\n")
for windc, row in per_windc.head(n_top).iterrows():
    line = f"  {windc:6s}  →  {row['n_oecd']} codes OECD"
    if show_codes:
        line += f"   {sorted(row['oecd_codes'])}"
    print(line)


In [ ]:
sector_labels = dict(zip(list(_s), _s.assoc_texts))
print("WiNDC without OECD — official BEA descriptions:")
for code in sorted(no_oecd):
    print(f"  {code:6s} : {sector_labels.get(code, '—')}")


## Final WiNDC ↔ OECD ICIO correspondence table

Complete table of the 71 matched WiNDC sectors. The sectors with several OECD codes reflect a real ambiguity in the NAICS/ISIC concordance (e.g. `con`: construction also covers the installation of industrial machinery in ISIC). The table is saved as CSV.

In [ ]:
_sector_labels_full = dict(zip(list(_s), _s.assoc_texts))
_bea_descs = (windc_summary_df
    .merge(BEAsummary_NAICS[['bea_summary','bea_summary_desc']].drop_duplicates(),
           on='bea_summary', how='left')
    [['windc_sector','bea_summary','bea_summary_desc']].drop_duplicates())

correspondance = (pairs[['windc_sector','oecd_code','OECD ICIO Industry']]
    .drop_duplicates()
    .merge(_bea_descs, on='windc_sector', how='left')
    .assign(windc_desc=lambda df: df['windc_sector'].map(_sector_labels_full))
    [['windc_sector','windc_desc','bea_summary','bea_summary_desc','oecd_code','OECD ICIO Industry']]
    .sort_values(['windc_sector','oecd_code'])
    .reset_index(drop=True))

correspondance.to_csv(
    str(ROOT / 'data/interim/windc_oecd_concordance.csv'),
    index=False
)
print(f'{len(correspondance)} rows | {correspondance["windc_sector"].nunique()} WiNDC sectors | '
      f'{correspondance["oecd_code"].nunique()} codes OECD couverts')
correspondance